In [ ]:
"""
Merge NASA Power Data with IMD Data
IMPORTANT: station1.csv must correspond to Station 1 (Anantwadi) in IMD file
           station2.csv must correspond to Station 2 (Arni) in IMD file
           ... and so on
"""

import pandas as pd
import numpy as np
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("NASA + IMD DATA MERGER")
print("="*80)

# ============================================================================
# STEP 1: LOAD IMD DATA
# ============================================================================

print("\nSTEP 1: Loading IMD data...")

imd_file = '/content/imd_data_all_stations_2018_2025_merged.xlsx'

try:
    imd_data = pd.read_excel(imd_file)
    print(f"IMD data loaded: {len(imd_data):,} rows")
    print(f"   Date range: {imd_data['date'].min()} to {imd_data['date'].max()}")
    print(f"   Stations: {imd_data['station_id'].nunique()}")

    # Show station order
    print("\n   Station order in IMD file:")
    for station_id in sorted(imd_data['station_id'].unique(), key=lambda x: int(x.split()[1])):
        station_data = imd_data[imd_data['station_id'] == station_id]
        station_name = station_data['station_name'].iloc[0]
        print(f"      {station_id}: {station_name}")

except Exception as e:
    print(f"Error loading IMD file: {e}")
    exit()

# ============================================================================
# STEP 2: LOAD NASA DATA (15 CSV FILES)
# ============================================================================

print("\n" + "="*80)
print("STEP 2: Loading NASA data from 15 CSV files...")
print("="*80)

nasa_data_list = []

for station_num in range(1, 16):
    csv_file = f'/content/station{station_num}.csv'

    # Get corresponding station name from IMD data
    station_id = f'Station {station_num}'
    station_info = imd_data[imd_data['station_id'] == station_id]

    if len(station_info) > 0:
        station_name = station_info['station_name'].iloc[0]
        print(f"\n{station_id}: {station_name:20s}")
    else:
        print(f"\n{station_id}: UNKNOWN")

    print(f"   File: {csv_file}")

    try:
        # Read CSV, skip first 15 rows (data starts at row 16)
        df = pd.read_csv(csv_file, skiprows=15)

        # Check if required columns exist
        required_cols = ['YEAR', 'DOY', 'T2M', 'RH2M', 'PS', 'WS2M', 'GWETTOP', 'GWETROOT', 'GWETPROF']
        missing_cols = [col for col in required_cols if col not in df.columns]

        if missing_cols:
            print(f"   ERROR: Missing columns: {missing_cols}")
            continue

        # Select only needed columns
        df = df[required_cols].copy()

        # Create date from YEAR and DOY for filtering
        df['temp_date'] = pd.to_datetime(df['YEAR'].astype(str) + '-' + df['DOY'].astype(str), format='%Y-%j')

        # Filter: Only keep data from 2018-06-01 to 2025-09-30
        df = df[(df['temp_date'] >= '2018-06-01') & (df['temp_date'] <= '2025-09-30')].copy()

        # Drop temporary date column
        df = df.drop('temp_date', axis=1)

        # CRITICAL: Add station identifier that matches IMD file
        df['station_id'] = station_id

        # Remove any rows with missing data in NASA parameters
        before_drop = len(df)
        df = df.dropna(subset=['T2M', 'RH2M', 'PS', 'WS2M', 'GWETTOP', 'GWETROOT', 'GWETPROF'])
        after_drop = len(df)

        if before_drop - after_drop > 0:
            print(f"   Removed {before_drop - after_drop} rows with missing NASA data")

        print(f"   Loaded: {len(df):,} records (2018-06-01 to 2025-09-30)")

        nasa_data_list.append(df)

    except FileNotFoundError:
        print(f"   ERROR: File not found!")
        print(f"   Please upload {csv_file} to Colab")
    except Exception as e:
        print(f"   ERROR: {e}")

if not nasa_data_list:
    print("\nNo NASA data loaded! Exiting...")
    exit()

# Combine all NASA data
nasa_data = pd.concat(nasa_data_list, ignore_index=True)
print(f"\n{'='*80}")
print(f"Total NASA records loaded: {len(nasa_data):,}")
print(f"Stations loaded: {nasa_data['station_id'].nunique()}")

# ============================================================================
# STEP 3: PREPARE DATA FOR MERGING
# ============================================================================

print("\n" + "="*80)
print("STEP 3: Preparing data for merging...")
print("="*80)

# Convert IMD date to datetime
imd_data['date'] = pd.to_datetime(imd_data['date'])
imd_data['YEAR'] = imd_data['date'].dt.year
imd_data['DOY_calc'] = imd_data['date'].dt.dayofyear

# Use existing 'doy' column if available, otherwise use calculated
if 'doy' in imd_data.columns:
    print("Using existing 'doy' column from IMD data")
else:
    imd_data['doy'] = imd_data['DOY_calc']
    print("Created 'doy' column from date")

print(f"\nIMD data ready:")
print(f"   Rows: {len(imd_data):,}")
print(f"   Date range: {imd_data['date'].min()} to {imd_data['date'].max()}")
print(f"   Stations: {imd_data['station_id'].nunique()}")

print(f"\nNASA data ready:")
print(f"   Rows: {len(nasa_data):,}")
print(f"   Stations: {nasa_data['station_id'].nunique()}")

# ============================================================================
# STEP 4: MERGE NASA WITH IMD
# ============================================================================

print("\n" + "="*80)
print("STEP 4: Merging NASA data with IMD data...")
print("="*80)

# Show merge key information
print("\nMerge keys:")
print("   - YEAR (from date)")
print("   - DOY (day of year)")
print("   - station_id (Station 1, Station 2, ...)")

# Merge on YEAR, DOY, and station_id
merged_data = imd_data.merge(
    nasa_data,
    left_on=['YEAR', 'doy', 'station_id'],
    right_on=['YEAR', 'DOY', 'station_id'],
    how='left',
    indicator=True
)

# Check merge statistics
merge_stats = merged_data['_merge'].value_counts()
print(f"\nMerge statistics:")
print(f"   Both (matched): {merge_stats.get('both', 0):,}")
print(f"   Left only (IMD no NASA): {merge_stats.get('left_only', 0):,}")
print(f"   Right only (NASA no IMD): {merge_stats.get('right_only', 0):,}")

# Drop merge indicator
merged_data = merged_data.drop('_merge', axis=1)

print(f"\nMerge complete: {len(merged_data):,} rows")

# Check for missing NASA data
nasa_cols = ['T2M', 'RH2M', 'PS', 'WS2M', 'GWETTOP', 'GWETROOT', 'GWETPROF']
missing_nasa = merged_data[nasa_cols].isnull().sum()

if missing_nasa.sum() > 0:
    print(f"\nWarning: Some NASA data missing:")
    for col, count in missing_nasa.items():
        if count > 0:
            print(f"   {col}: {count:,} missing values ({count/len(merged_data)*100:.2f}%)")
else:
    print("\nPERFECT! All NASA data matched with IMD data!")

# ============================================================================
# STEP 5: REORDER COLUMNS
# ============================================================================

print("\n" + "="*80)
print("STEP 5: Reordering columns...")
print("="*80)

# Final column order
final_columns = [
    'date',
    'doy',
    'T2M',
    'RH2M',
    'PS',
    'WS2M',
    'GWETTOP',
    'GWETROOT',
    'GWETPROF',
    'rain',
    'tmin',
    'tmax',
    'station_name',
    'station_id'
]

# Select only these columns
merged_data = merged_data[final_columns].copy()

# Sort by station_id (numerically) and date
merged_data['station_num'] = merged_data['station_id'].str.extract('(\d+)').astype(int)
merged_data = merged_data.sort_values(['station_num', 'date']).reset_index(drop=True)
merged_data = merged_data.drop('station_num', axis=1)

print(f"Columns reordered")
print(f"Final columns: {list(merged_data.columns)}")

# ============================================================================
# STEP 6: SAVE TO EXCEL
# ============================================================================

print("\n" + "="*80)
print("STEP 6: Saving merged data to Excel...")
print("="*80)

output_file = '/content/merged_imd_nasa_data_2018_2025.xlsx'

try:
    # Convert date to date only (remove time)
    merged_data['date'] = pd.to_datetime(merged_data['date']).dt.date

    merged_data.to_excel(output_file, index=False, sheet_name='Merged_Data')

    print(f"\nExcel file saved: {output_file}")
    print(f"File size: {os.path.getsize(output_file) / (1024*1024):.2f} MB")

except Exception as e:
    print(f"Error saving Excel: {e}")
    exit()

# ============================================================================
# STEP 7: DISPLAY SUMMARY
# ============================================================================

print("\n" + "="*80)
print("MERGE SUMMARY")
print("="*80)

print(f"\nTotal records: {len(merged_data):,}")
print(f"Date range: {merged_data['date'].min()} to {merged_data['date'].max()}")
print(f"Total stations: {merged_data['station_id'].nunique()}")

print(f"\nSample data (first 10 rows):")
print(merged_data.head(10).to_string(index=False))

print(f"\nSample data (last 10 rows):")
print(merged_data.tail(10).to_string(index=False))

# Per station statistics
print(f"\n" + "="*80)
print("PER STATION SUMMARY")
print("="*80)

for station_id in sorted(merged_data['station_id'].unique(), key=lambda x: int(x.split()[1])):
    station_data = merged_data[merged_data['station_id'] == station_id]
    station_name = station_data['station_name'].iloc[0]

    # Count missing NASA values
    missing_count = station_data[nasa_cols].isnull().any(axis=1).sum()

    # Count complete rows
    complete_count = len(station_data) - missing_count

    print(f"\n{station_id}: {station_name}")
    print(f"   Total records: {len(station_data):,}")
    print(f"   Complete rows: {complete_count:,}")
    print(f"   Missing NASA: {missing_count}")
    print(f"   Completeness: {(complete_count/len(station_data)*100):.2f}%")

# Overall data completeness
print(f"\n" + "="*80)
print("OVERALL DATA QUALITY")
print("="*80)

total_cells = len(merged_data) * len(final_columns)
missing_cells = merged_data.isnull().sum().sum()
completeness = ((total_cells - missing_cells) / total_cells) * 100

print(f"\nTotal cells: {total_cells:,}")
print(f"Missing cells: {missing_cells:,}")
print(f"Completeness: {completeness:.2f}%")

print("\n" + "="*80)
print("MERGE COMPLETE!")
print("="*80)

print(f"\nOutput file: {output_file}")

print("\nDone! Your NASA + IMD data is properly merged!")

<>:230: SyntaxWarning: invalid escape sequence '\d'
<>:230: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-680523370.py:230: SyntaxWarning: invalid escape sequence '\d'
  merged_data['station_num'] = merged_data['station_id'].str.extract('(\d+)').astype(int)


NASA + IMD DATA MERGER

STEP 1: Loading IMD data...
IMD data loaded: 40,185 rows
   Date range: 2018-06-01 00:00:00 to 2025-09-30 00:00:00
   Stations: 15

   Station order in IMD file:
      Station 1: Anantwadi
      Station 2: Arni
      Station 3: Khadka
      Station 4: Koli(Bk)
      Station 5: Murli
      Station 6: Sharad
      Station 7: Takali
      Station 8: Bhamragad
      Station 9: Hamdapur
      Station 10: Kunturla_Machkund
      Station 11: Ankushapur
      Station 12: Garimillapalli
      Station 13: AllamVariGhanapur
      Station 14: Gandlapet
      Station 15: Valamuru_Pamuleru

STEP 2: Loading NASA data from 15 CSV files...

Station 1: Anantwadi           
   File: /content/station1.csv
   Loaded: 2,679 records (2018-06-01 to 2025-09-30)

Station 2: Arni                
   File: /content/station2.csv
   Loaded: 2,679 records (2018-06-01 to 2025-09-30)

Station 3: Khadka              
   File: /content/station3.csv
   Loaded: 2,679 records (2018-06-01 to 2025-09-3